## Convert Open Images Threat to Yolo Annotations

In [ ]:
import os
import fiftyone as fo

# Load your OpenImages dataset
dataset = fo.load_dataset("open-images-threat-detection")
dataset.compute_metadata()

# Define your final kept classes and assign YOLO class IDs
final_classes = [
    "Truck", "Baseball bat", "Rifle", "Weapon", "Knife", "Missile", "Sword",
    "Shotgun", "Handgun", "Backpack", "Dagger", "Bow and arrow", "Suitcase", "Scissors"
]

class_to_id = {cls: idx for idx, cls in enumerate(final_classes)}

# Path where YOLO annotations will be saved
output_labels_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/openimages-threat/labels_temp/"
os.makedirs(output_labels_dir, exist_ok=True)

# Conversion function
def convert_to_yolo(bbox, img_width, img_height):
    """
    Converts a bounding box from (xmin, ymin, xmax, ymax) to YOLO format (x_center, y_center, width, height),
    all normalized to [0, 1].
    """
    xmin, ymin, xmax, ymax = bbox
    x_center = (xmin + xmax) / 2 / img_width
    y_center = (ymin + ymax) / 2 / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height
    return x_center, y_center, width, height

# Process each sample
for sample in dataset:
    img_metadata = sample.metadata
    if img_metadata is None:
        print(f"Skipping {sample.filepath}: missing metadata")
        continue

    img_width = img_metadata.width
    img_height = img_metadata.height

    label_lines = []

    detections = sample.ground_truth.detections
    for det in detections:
        label = det.label
        if label not in final_classes:
            continue

        bbox = det.bounding_box  # [x, y, width, height] normalized [0, 1]
        
        # If bounding box is already normalized, just work with it
        xmin = bbox[0] * img_width
        ymin = bbox[1] * img_height
        xmax = xmin + bbox[2] * img_width
        ymax = ymin + bbox[3] * img_height

        # Convert to YOLO format
        x_center, y_center, width, height = convert_to_yolo((xmin, ymin, xmax, ymax), img_width, img_height)

        # Map class name to YOLO id
        class_id = class_to_id[label]

        # YOLO format line
        yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        label_lines.append(yolo_line)

    # Write .txt file if there are relevant labels
    if label_lines:
        # Save label file with the same name as image, just .txt
        img_name = os.path.splitext(os.path.basename(sample.filepath))[0]
        label_file = os.path.join(output_labels_dir, f"{img_name}.txt")

        with open(label_file, "w") as f:
            f.write("\n".join(label_lines))

Computing metadata...
 100% |█████████████| 19174/19174 [1.9s elapsed, 0s remaining, 10.5K samples/s]      
